# [LAB12] 딥러닝 > 신경망의 이해 > 07. 다중선형회귀 + 하이퍼파라미터 튜닝

농어의 무게 예측 데이터 셋

keras-tuner 패키지의 설치가 필요하다.

## 📘 #01. 준비작업

### 📝 [1] 패키지 가져오기

In [ ]:
!pip install --upgrade keras-tuner

In [ ]:
from hossam import *
from pandas import DataFrame
from matplotlib import pyplot as plt
import seaborn as sb
import numpy as np
from datetime import datetime as dt
from keras_tuner import Hyperband
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD, RMSprop
from tensorflow.keras.losses import mse
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import RootMeanSquaredError, R2Score
from tqdm.keras import TqdmCallback

### 📝 [2] 데이터셋 준비하기

### 📝 [3] 랜덤시드 고정

## 📘 #02. 탐색적 데이터 분석

### 📝 [1] 데이터 품질 검사

In [ ]:
origin = load_data("fish")
origin.head()

In [ ]:
np.random.seed(52)
desc = origin.describe().T
num_cols = origin.select_dtypes(include=np.number).columns
for column in num_cols:
    skewness = origin[column].skew()
    if abs(skewness) < 0.5:
        strength = "week"
        log_transform = "not needed"
    elif abs(skewness) < 1:
        strength = "normal"
        log_transform = "recommended"
    else:
        strength = "strong"
        log_transform = "needed"
    desc.loc[column, "skewness"] = skewness
    desc.loc[column, "skewness_strength"] = strength
    desc.loc[column, "log_transform"] = log_transform
desc

### 📝 [3] 무게에 대한 로그 변환

## 📘 #03. 데이터 전처리

### 📝 [1] 훈련/검증 데이터 분리

### 📝 [2] 다중 공선성 제거 (예제 데이터셋이므로 다중공선성은 고려하지 않는다)

In [ ]:
df = origin.copy()
df['무게'] = np.log1p(df['무게'])
df.head()

In [ ]:
yname = "무게"
x = df.drop(columns=[yname])
y = df[yname]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=52)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

## 📘 #04. 신경망 모델 적합

### 📝 [1] 하이퍼파라미터 튜닝 정의

튜닝할 파라미터를 설정하는 콜백함수를 정의해야 한다.

In [ ]:
_, cols = x_train.shape
def tf_build(hp) -> Sequential:
    model = Sequential()
    model.add(Input(shape=(cols,)))
    model.add(Dense(units=hp.Choice("units", values=[4, 8, 16, 32, 64]), activation="relu"))
    model.add(Dense(1, activation="linear"))
    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae", RootMeanSquaredError(name="rmse"), R2Score(name="r2")],
    )
    return model

### 📝 [2] 튜너 객체 생성

### 📝 [3] 하이퍼파라미터 튜닝 수행

In [ ]:
tuner = Hyperband(
    hypermodel=tf_build,
    objective="val_mae",
    max_epochs=10,
    factor=3,
    seed=52,
    directory="D:\\tensor_hyperband",
    project_name="tf_hyperband_%s" % dt.now().strftime("%Y%m%d%H%M%S"),
)
tuner

In [ ]:
tuner.search(
    x_train, y_train, epochs=10, batch_size=32, validation_data=(x_test, y_test)
)
best_hps = tuner.get_best_hyperparameters()
if not best_hps:
    raise ValueError("No best hyperparameters found.")
print(f"""best hyperparameters: {best_hps[0].values}""")

### 📝 [4] 최종 모형 도출

도출된 하이퍼 파라미터가 적용된 새로운 모델을 생성하고, 학습을 별도로 수행해야 한다.

## 📘 #05 성능평가

### 📝 [1] 성능평가 지표

### 📝 [2] 학습 과정 확인

### 📝 [3] Loss, RMSE 학습곡선

In [ ]:
model = tuner.hypermodel.build(best_hps[0])
result = model.fit(
    x_train, y_train,
    epochs=500,
    validation_data=(x_test, y_test),
    verbose=0,
    callbacks=[
        TqdmCallback(verbose=1),
        EarlyStopping(monitor='val_loss', patience=5, min_delta=0.001),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=0, verbose=1)
    ]
)
result

In [ ]:
train_eval = model.evaluate(x_train, y_train, verbose=0, return_dict=True)
test_eval = model.evaluate(x_test, y_test, verbose=0, return_dict=True)
final_results = DataFrame([train_eval, test_eval])
final_results.insert(0, "Dataset", ["Train", "Test"])
final_results["RMSE_gap"] = None
final_results.loc[1, "RMSE_gap"] = final_results.loc[1, "rmse"] - final_results.loc[0, "rmse"]
final_results

In [ ]:
history_df = DataFrame(data=result.history)
history_df["epoch"] = history_df.index + 1
history_df.head()

In [ ]:
figsize = (1600 / 100, 600 / 100)
fig, ax = plt.subplots(1, 2, figsize=figsize, dpi=100)
fig.subplots_adjust(wspace=0.2, hspace=0.2)
sb.lineplot(data=history_df, x="epoch", y="loss", ax=ax[0], label="Train Loss")
sb.lineplot(data=history_df, x="epoch", y="val_loss", ax=ax[0], label="Validation Loss")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].set_title("Training vs Validation Loss")
ax[0].grid(True, alpha=0.3)
sb.lineplot(data=history_df, x="epoch", y="rmse", ax=ax[1], label="Train RMSE")
sb.lineplot(data=history_df, x="epoch", y="val_rmse", ax=ax[1], label="Validation RMSE")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("RMSE")
ax[1].set_title("Training vs Validation RMSE")
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

## 📘 #06. 예측 결과 활용

### 📝 [1] 예측치 구하기

### 📝 [2] 결과 데이터 셋 구성

### 📝 [3] 관측치와 예측치 비교 시각화

In [ ]:
pred = model.predict(x_test, verbose=0)
pred[:5]

In [ ]:
kdf = DataFrame({
    '길이': x_test['길이'],
    '실제값': y_test,
    '예측값': pred.flatten()
})
kdf['오차'] = kdf['실제값']-kdf['예측값']
kdf.head()

In [ ]:
figsize = (1280 / 100, 720 / 100)
fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=100)
sb.regplot(data=kdf, x='길이', y='실제값', label='실제값')
sb.regplot(data=kdf, x='길이', y='예측값', label='예측값')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title("실제값 vs 예측값")
plt.tight_layout()
plt.show()
plt.close()